<a href="https://colab.research.google.com/github/Thcastro2004/ECSE551-A2-ML-for-engineers/blob/main/Barnett_Cottereau_Zhang_Assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 1 - CNN

In [1]:
#import statements
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import numpy as np
import csv
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, Subset, DataLoader
from sklearn.model_selection import train_test_split, KFold
from PIL import Image
import os
import zipfile
from kaggle.api.kaggle_api_extended import KaggleApi

ModuleNotFoundError: No module named 'torch'

Download the Kaggle dataset

In [ ]:
# Download Kaggle competition data if not already present
if not os.path.exists('./data/train') or not os.path.exists('./data/test'):
    api = KaggleApi()
    api.authenticate()
    os.makedirs('./data', exist_ok=True)
    
    competition_name = 'ecse-551-assignment-2-task-2'
    api.competition_download_files(competition_name, path='./data')
    
    zip_path = f'./data/{competition_name}.zip'
    if os.path.exists(zip_path):
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall('./data/')
        os.remove(zip_path)

Importing and pre-processing data using a custom Dataset for the training dataset

In [ ]:
class TrainingDataset(Dataset): #10000 samples with labels, performs random data augmentation
  def __init__(self):
    super().__init__()
    self.df = pd.read_csv('./data/kaggle_challenge/train_labels.csv')
    self.trainpath = './data/kaggle_challenge/train/'
    self.transform = transforms.Compose([
        transforms.RandomApply(nn.ModuleList([transforms.RandomResizedCrop(size=(32,32), scale=(0.8,1))]),p=0.1),
        transforms.RandomApply(nn.ModuleList([transforms.RandomRotation((1,5))]),p=0.1),
        transforms.RandomHorizontalFlip(p=0.1),
        transforms.RandomApply(nn.ModuleList([transforms.ColorJitter((0.7,1),(0.7,1),(0.7,1),(-0.1,0.1))]),p=0.1),
        transforms.ToTensor(), #converts a PIL image to a Pytorch Tensor
        transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
        ])
    self.labels_dict = {
    "truck": 0,
    "deer": 1,
    "bird": 2,
    "frog": 3,
    "ship": 4,
    "horse": 5,
    "cat": 6,
    "dog": 7,
    "automobile": 8,
    "airplane": 9
}
    return

  def __getitem__(self, idx):
    img = Image.open(self.trainpath+self.df.at[idx, "id"])
    img = img.convert("RGB")
    img = self.transform(img)
    label = self.labels_dict.get(self.df.at[idx, "label"])
    return img, label

  def __len__(self):
    return len(self.df)

In [ ]:
class TestingDataset(Dataset): #10000 samples with labels, no transformations
  def __init__(self):
    super().__init__()
    self.df = pd.read_csv('./data/kaggle_challenge/train_labels.csv')
    self.trainpath = './data/kaggle_challenge/train/'
    self.transform = transforms.Compose([
        transforms.ToTensor(), #converts a PIL image to a Pytorch Tensor
        transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
        ])
    self.labels_dict = {
    "truck": 0,
    "deer": 1,
    "bird": 2,
    "frog": 3,
    "ship": 4,
    "horse": 5,
    "cat": 6,
    "dog": 7,
    "automobile": 8,
    "airplane": 9
  }
    return

  def __getitem__(self, idx):
    img = Image.open(self.trainpath+self.df.at[idx, "id"])
    img = img.convert("RGB")
    img = self.transform(img)
    label = self.labels_dict.get(self.df.at[idx, "label"])
    return img, label

  def __len__(self):
    return len(self.df)

CNN Class Definition

In [ ]:
class CNN(nn.Module):

  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
    self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
    self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
    self.conv4 = nn.Conv2d(64, 128, 3, padding=1)
    self.conv5 = nn.Conv2d(128, 256, 4, padding=1)
    self.conv6 = nn.Conv2d(256, 512, 4, padding=1)
    self.pool = nn.MaxPool2d(2, 2)
    self.lin1 = nn.Linear(8192, 4096)
    self.lin2 = nn.Linear(4096, 512)
    self.lin3 = nn.Linear(512, 128)
    self.lin4 = nn.Linear(128, 10)
    self.batchnorm1 = nn.BatchNorm2d(16)
    self.batchnorm2 = nn.BatchNorm2d(32)
    self.batchnorm3 = nn.BatchNorm2d(64)
    self.batchnorm4 = nn.BatchNorm2d(128)
    self.batchnorm5 = nn.BatchNorm2d(256)
    self.batchnorm6 = nn.BatchNorm2d(512)
    self.dropout = nn.Dropout(0.3)
    return

  def forward(self, x):
    x = nn.functional.relu(self.batchnorm1(self.conv1(x)))
    x = nn.functional.relu(self.batchnorm2(self.conv2(x)))
    x = self.pool(x)
    x = nn.functional.relu(self.batchnorm3(self.conv3(x)))
    x = nn.functional.relu(self.batchnorm4(self.conv4(x)))
    x = self.pool(x)
    # x = nn.functional.relu(self.batchnorm5(self.conv5(x)))
    # x = nn.functional.relu(self.batchnorm6(self.conv6(x)))
    # x = self.pool(x)
    x = torch.flatten(x, 1)
    x = nn.functional.relu(self.lin1(x))
    x = self.dropout(x)
    x = nn.functional.relu(self.lin2(x))
    x = self.dropout(x)
    x = nn.functional.relu(self.lin3(x))
    x = self.dropout(x)
    x = self.lin4(x)
    return x

CNN Training

In [ ]:
#split training data
training_dataset = TrainingDataset() #initiate Dataset
testing_dataset = TestingDataset()

indices = list(range(len(training_dataset)))
train_indices, test_indices = train_test_split(indices, test_size=0.3, random_state=42)
train_dataset = Subset(training_dataset, train_indices)
validation_dataset = Subset(testing_dataset, train_indices)
final_test_dataset = Subset(testing_dataset, test_indices) #for final model performance analysis

results = []
weights = []
kf = KFold(shuffle=True, random_state=42)

for i, (train_idx, test_idx) in enumerate(kf.split(train_indices)):
  cnn = CNN()
  cnn.train()
  loss_function = nn.CrossEntropyLoss()
  optimizer = torch.optim.Adam(cnn.parameters(), 0.0003, weight_decay=0.0001)

  kf_train_dataset = Subset(train_dataset, train_idx)
  kf_train_dataloader = DataLoader(kf_train_dataset, batch_size=64, shuffle=True, num_workers=2)
  kf_test_dataset = Subset(validation_dataset, test_idx)
  kf_test_dataloader = DataLoader(kf_test_dataset, batch_size=64, shuffle=False, num_workers=2)

  training_loss = []
  for epoch in range(5):
    for input, label in kf_train_dataloader:
      output = cnn(input)
      loss = loss_function(output, label)
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()
      training_loss.append(loss.item())
      print("batch loss:"+str(loss.item()))
    print(f"epoch {epoch}: {loss.item()}")

  plt.plot(training_loss)
  plt.xlabel("Batch number")
  plt.ylabel("Loss")
  plt.title("Batch-wise Training Loss")
  plt.show()
  fold_filename = f"./cnn_fold_{i+1}.pth"
  torch.save(cnn.state_dict(), fold_filename)

  cnn.eval()
  with torch.no_grad():
    correct = 0
    for input, label in kf_test_dataloader:
      output = cnn(input)
      prediction = output.argmax(dim=1)
      correct += (prediction == label).sum().item()
  results.append(correct/len(test_idx))
  print(results)

Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/Users/thomascottereau/opt/anaconda3/lib/python3.9/multiprocessing/spawn.py", line 116, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/Users/thomascottereau/opt/anaconda3/lib/python3.9/multiprocessing/spawn.py", line 126, in _main
    self = reduction.pickle.load(from_parent)
AttributeError: Can't get attribute 'TrainingDataset' on <module '__main__' (built-in)>


KeyboardInterrupt: 

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# Commented out - not needed for local execution

Mounted at /content/drive


CNN Performance Evaluation

Creating a Dataset for the 2000 unlabelled images for Kaggle

In [ ]:
class Test551(Dataset): #2000

  def __init__(self):
    super().__init__()
    self.df = pd.read_csv('./data/kaggle_challenge/sample_submission.csv')
    self.trainpath = './data/kaggle_challenge/test/'
    self.transform = transforms.ToTensor() #converts a PIL image to a Pytorch Tensor
    self.labels_dict = {
    "truck": 0,
    "deer": 1,
    "bird": 2,
    "frog": 3,
    "ship": 4,
    "horse": 5,
    "cat": 6,
    "dog": 7,
    "automobile": 8,
    "airplane": 9
    }
    return

  def __getitem__(self, idx):
    img = Image.open(self.trainpath+self.df.at[idx, "id"])
    img = img.convert("RGB")
    img = self.transform(img)
    label = 0 #no labels assigned since this is the test set
    return img, label

  def __len__(self):
    return len(self.df)

CNN Prediction for Kaggle

In [ ]:

kaggle_dataset = Test551()
kaggle_dataloader = DataLoader(kaggle_dataset, batch_size=64, shuffle=True, num_workers=2)


for input, label in kaggle_dataloader:
  output =



In [ ]:
conv1 = nn.Conv2d(3, 16, 3, padding=1)
conv2 = nn.Conv2d(16, 32, 3, padding=1)
pool = nn.MaxPool2d(2, 2)
conv3 = nn.Conv2d(32, 64, 4, padding=1)
conv4 = nn.Conv2d(64, 128, 4, padding=1)
conv5 = nn.Conv2d(128, 256, 3, padding=1)
conv6 = nn.Conv2d(256, 512, 3, padding=1)
lin1 = nn.Linear(8192, 512)
lin2 = nn.Linear(512, 128)
lin3 = nn.Linear(128, 10)
batchnorm1 = nn.BatchNorm2d(16)
batchnorm2 = nn.BatchNorm2d(32)
batchnorm3 = nn.BatchNorm2d(64)
batchnorm4 = nn.BatchNorm2d(128)
batchnorm5 = nn.BatchNorm2d(256)
batchnorm6 = nn.BatchNorm2d(512)

x = torch.randn(1, 3, 32, 32)
x = nn.functional.relu(batchnorm1(conv1(x)))
x = nn.functional.relu(batchnorm2(conv2(x)))
x = pool(x)
x = nn.functional.relu(batchnorm3(conv3(x)))
x = nn.functional.relu(batchnorm4(conv4(x)))
x = pool(x)
# x = nn.functional.relu(conv5(x))
# x = nn.functional.relu(conv6(x))
# x = pool(x)
x = torch.flatten(x, 1)

flattened_size = x.shape[1]
print(flattened_size)
# x = nn.functional.relu(lin1(x))
# x = nn.functional.relu(lin2(x))
# x = lin3(x)


6272


# Task 1 - Vision Transformer

In [ ]:
dataset = Dataset551()


# Task 2 - Free for All

In [ ]:
# Import additional libraries for Task 2
import torchvision.models as models
from torch.optim.lr_scheduler import StepLR, CosineAnnealingLR
import torch.nn.functional as F


## Task 2 Additional Imports

Imports additional libraries needed for Task 2:
- **torchvision.models**: Pre-trained models (ResNet, EfficientNet, ViT, etc.)
- **Learning rate schedulers**: StepLR and CosineAnnealingLR for better training
- **torch.nn.functional**: Additional neural network functions


## Task 2 Strategy Options:

1. **Transfer Learning with Pre-trained Models**: Use ResNet, EfficientNet, or Vision Transformer pre-trained on ImageNet
2. **Ensemble Methods**: Combine predictions from multiple models (your Task 1 models + pre-trained models)
3. **Advanced Data Augmentation**: Mixup, CutMix, AutoAugment
4. **Better Training**: Longer training, learning rate scheduling, better optimizers

### Recommended Approach:
Start with a pre-trained ResNet50 or EfficientNet, fine-tune on your dataset, then optionally ensemble with your Task 1 models.


## Pre-trained Model Creation Functions

Helper functions to create transfer learning models:
- **create_resnet_model()**: ResNet50 pre-trained on ImageNet, fine-tuned for 10 classes
- **create_efficientnet_model()**: EfficientNet-B3, a more efficient architecture
- **create_vit_model()**: Vision Transformer pre-trained on ImageNet

All models replace the final classification layer to match our 10-class problem.


## Task 2 Dataset Classes

Enhanced dataset classes for Task 2 with more aggressive data augmentation:

**Task2TrainingDataset**:
- Stronger augmentation (random crops, rotations, flips, color jitter, affine transforms)
- ImageNet normalization statistics (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
- Higher augmentation probabilities for better generalization

**Task2TestDataset**:
- Minimal transformations for validation
- Same ImageNet normalization for consistency with pre-trained models


## Task 2 Training with Transfer Learning

Complete training pipeline for Task 2:
1. **Data preparation**: 80% train, 20% validation split
2. **Model setup**: ResNet50 with pre-trained ImageNet weights
3. **Training configuration**:
   - AdamW optimizer (lr=1e-4, weight_decay=1e-4)
   - Cosine annealing learning rate scheduler
   - 50 epochs
4. **Training loop**: Tracks training loss and validation accuracy
5. **Model checkpointing**: Saves the best model based on validation accuracy

This approach leverages transfer learning to achieve better performance with less training time.


In [ ]:
# Example: Transfer Learning with ResNet50
# You can use ANY pre-trained model here (ResNet, EfficientNet, ViT, etc.)

def create_resnet_model(num_classes=10):
    # Load pre-trained ResNet50
    model = models.resnet50(weights='IMAGENET1K_V2')
    
    # Replace the final fully connected layer for 10 classes
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, num_classes)
    
    return model

# Alternative: EfficientNet
def create_efficientnet_model(num_classes=10):
    model = models.efficientnet_b3(weights='IMAGENET1K_V1')
    num_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(num_features, num_classes)
    return model

# Alternative: Vision Transformer (pre-trained)
def create_vit_model(num_classes=10):
    model = models.vit_b_16(weights='IMAGENET1K_V1')
    num_features = model.heads.head.in_features
    model.heads.head = nn.Linear(num_features, num_classes)
    return model


## Task 2 Kaggle Submission Generation

Generates predictions for the Kaggle competition:
1. Loads the best trained model from Task 2
2. Runs inference on the test dataset
3. Converts class indices to label names
4. Creates a submission CSV file in the format required by Kaggle

The submission file can be directly uploaded to the competition leaderboard.


In [ ]:
# Enhanced data augmentation for Task 2
class Task2TrainingDataset(Dataset):
    def __init__(self):
        super().__init__()
        self.df = pd.read_csv('./data/kaggle_challenge/train_labels.csv')
        self.trainpath = './data/kaggle_challenge/train/'
        # More aggressive augmentation for Task 2
        self.transform = transforms.Compose([
            transforms.RandomResizedCrop(size=(32, 32), scale=(0.7, 1.0)),
            transforms.RandomRotation(degrees=15),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.1),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet stats
        ])
        self.labels_dict = {
            "truck": 0, "deer": 1, "bird": 2, "frog": 3, "ship": 4,
            "horse": 5, "cat": 6, "dog": 7, "automobile": 8, "airplane": 9
        }
    
    def __getitem__(self, idx):
        img = Image.open(self.trainpath + self.df.at[idx, "id"])
        img = img.convert("RGB")
        img = self.transform(img)
        label = self.labels_dict.get(self.df.at[idx, "label"])
        return img, label
    
    def __len__(self):
        return len(self.df)

class Task2TestDataset(Dataset):
    def __init__(self):
        super().__init__()
        self.df = pd.read_csv('./data/kaggle_challenge/train_labels.csv')
        self.trainpath = './data/kaggle_challenge/train/'
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        self.labels_dict = {
            "truck": 0, "deer": 1, "bird": 2, "frog": 3, "ship": 4,
            "horse": 5, "cat": 6, "dog": 7, "automobile": 8, "airplane": 9
        }
    
    def __getitem__(self, idx):
        img = Image.open(self.trainpath + self.df.at[idx, "id"])
        img = img.convert("RGB")
        img = self.transform(img)
        label = self.labels_dict.get(self.df.at[idx, "label"])
        return img, label
    
    def __len__(self):
        return len(self.df)


In [ ]:
# Task 2 Training Example with ResNet50
# You can modify this to use different models or create an ensemble

task2_train_dataset = Task2TrainingDataset()
task2_test_dataset = Task2TestDataset()

# Split data
indices = list(range(len(task2_train_dataset)))
train_indices, val_indices = train_test_split(indices, test_size=0.2, random_state=42)

train_subset = Subset(task2_train_dataset, train_indices)
val_subset = Subset(task2_test_dataset, val_indices)

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_subset, batch_size=32, shuffle=False, num_workers=2)

# Create model
model_task2 = create_resnet_model(num_classes=10)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_task2 = model_task2.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model_task2.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-6)

# Training loop
num_epochs = 50
best_val_acc = 0.0

for epoch in range(num_epochs):
    # Training
    model_task2.train()
    train_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model_task2(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    scheduler.step()
    
    # Validation
    model_task2.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model_task2(inputs)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    
    val_acc = val_correct / val_total
    print(f'Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss/len(train_loader):.4f}, Val Acc: {val_acc:.4f}')
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model_task2.state_dict(), './task2_best_model.pth')
        print(f'New best model saved! Val Acc: {best_val_acc:.4f}')

print(f'Training complete. Best validation accuracy: {best_val_acc:.4f}')


## Ensemble Example (Optional but Recommended)

You can combine predictions from multiple models to improve performance:
- Your Task 1 CNN
- Your Task 1 ViT  
- Pre-trained ResNet
- Pre-trained EfficientNet
- etc.

Average or weight their predictions for final submission.


In [ ]:
# Example: Generate predictions for Kaggle submission
# Load best model and predict on test set

kaggle_test_dataset = Test551()
kaggle_test_loader = DataLoader(kaggle_test_dataset, batch_size=32, shuffle=False, num_workers=2)

# Load best model
model_task2.load_state_dict(torch.load('./task2_best_model.pth'))
model_task2.eval()

predictions = []
label_names = ["truck", "deer", "bird", "frog", "ship", "horse", "cat", "dog", "automobile", "airplane"]

with torch.no_grad():
    for inputs, _ in kaggle_test_loader:
        inputs = inputs.to(device)
        outputs = model_task2(inputs)
        _, predicted = torch.max(outputs, 1)
        predictions.extend(predicted.cpu().numpy())

# Create submission file
submission_df = pd.read_csv('./data/kaggle_challenge/sample_submission.csv')
submission_df['label'] = [label_names[pred] for pred in predictions]
submission_df.to_csv('./task2_submission.csv', index=False)
print("Submission file created!")
